<img src="https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/UNIFOR_logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

Lattes: http://lattes.cnpq.br/5738786447903616 |
GitHub: https://github.com/carubbi/

[Unifor.br](https://unifor.br/) | [Instagram](https://www.instagram.com/uniforcomunica/?hl=pt-br) | [Facebook](https://www.facebook.com/uniforoficial/) | [Twitter](https://www.facebook.com/uniforoficial/)  | [LinkedIn](https://www.linkedin.com/school/university-of-fortaleza/?originalSubdomain=pt) | [TV Unifor](https://www.unifor.br/tv-unifor) | [G1/Ensinando e Aprendendo](https://g1.globo.com/ce/ceara/especial-publicitario/unifor/ensinando-e-aprendendo/)

# Resumo

## **Aula 2: Fundamentos estatísticos e investigação com dados**

Nesta aula, o dataset `penguins_raw.csv` será usado como uma **população finita de referência**. O objetivo não é generalizar para todos os pinguins, mas observar, com código, como diferentes procedimentos de seleção alteram os dados disponíveis para análise.


## Objetivos de aplicação

Ao final da aula, você deverá ser capaz de:

- formular uma pergunta estatística compatível com os dados disponíveis;
- reconhecer variabilidade nas observações;
- definir a população de referência e a unidade observacional usada na atividade;
- selecionar amostras aleatórias reproduzíveis com `sample()`;
- comparar a composição de duas amostras com a composição do conjunto de referência;
- reconhecer quando um filtro é adequado ao objetivo e quando produz viés de seleção;
- distinguir descrição e inferência;
- delimitar quais conclusões os dados selecionados permitem sustentar.


## Situação prática

Uma equipe recebeu as 344 observações do dataset Palmer Penguins e precisa testar procedimentos de amostragem antes de iniciar uma análise. Para esta atividade:

- **população finita de referência:** as 344 linhas disponíveis no arquivo;
- **unidade observacional:** um pinguim registrado em um estudo;
- **amostra:** um subconjunto dessas linhas.

Essa definição vale para o exercício computacional. O arquivo não deve ser tratado como se fosse a população de todos os pinguins das três espécies (ESCOVEDO; MARQUES; KALINOWSKI, 2025).


## Preparação instrumental dos dados

In [1]:
# Importar a biblioteca utilizada na aula.
import pandas as pd


In [2]:
# Definir a localização e carregar os dados.
arquivo_dados = "https://raw.githubusercontent.com/carubbi/MQ/main/data/raw/penguins_raw.csv"
df = pd.read_csv(arquivo_dados)

# Visualizar as primeiras observações.
df.head()


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


**Tabela 1 - Primeiras observações do dataset.** Cada linha registra um pinguim e cada coluna armazena uma característica ou informação do estudo. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

Para a atividade, serão mantidas apenas as colunas necessárias:

- `studyName`: estudo em que a observação foi registrada;
- `Individual ID`: identificador do indivíduo dentro do estudo;
- `Species`: espécie do pinguim;
- `Island`: ilha onde o pinguim foi observado;
- `Body Mass (g)`: massa corporal, em gramas.

Como o identificador pode se repetir entre estudos, a unidade é identificada pelo par `studyName` + `Individual ID` (HORST; HILL; GORMAN, 2020).


In [3]:
# Escolher as colunas necessárias.
colunas_amostras = [
    "studyName",
    "Individual ID",
    "Species",
    "Island",
    "Body Mass (g)",
]

# Criar a população finita de referência.
populacao_referencia = df[colunas_amostras].copy()

# Verificar o número de linhas e colunas.
populacao_referencia.shape


(344, 5)

A saída `(344, 5)` informa que a população finita de referência possui 344 linhas e 5 colunas selecionadas.

## Ciclo didático 1 — Investigação estatística e variabilidade

### Conhecer a composição da população de referência

In [4]:
# Contar quantas observações existem de cada espécie.
composicao_referencia = populacao_referencia["Species"].value_counts()
composicao_referencia


Species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64

**Tabela 2 - Composição da população de referência por espécie.** A tabela será o parâmetro de comparação para as amostras produzidas nesta aula. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

In [5]:
# Contar quantas observações existem em cada ilha.
composicao_ilhas = populacao_referencia["Island"].value_counts()
composicao_ilhas


Island
Biscoe       168
Dream        124
Torgersen     52
Name: count, dtype: int64

**Tabela 3 - Composição da população de referência por ilha.** As quantidades mostram que as ilhas não aparecem com a mesma frequência no arquivo. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

Antes de sortear, a equipe conhece a composição do conjunto disponível. Isso permite verificar depois quais grupos apareceram, desapareceram ou ficaram super-representados nas amostras.

In [6]:
# Resumir a massa corporal para reconhecer sua variabilidade.
resumo_massa = populacao_referencia["Body Mass (g)"].describe()
resumo_massa


count     342.000000
mean     4201.754386
std       801.954536
min      2700.000000
25%      3550.000000
50%      4050.000000
75%      4750.000000
max      6300.000000
Name: Body Mass (g), dtype: float64

**Tabela 4 - Resumo da massa corporal.** A saída descreve os valores disponíveis de `Body Mass (g)` e permite reconhecer quantidade válida, posição e variabilidade. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

**Registro do estudante:** formule uma pergunta respondível sobre a massa corporal e explique por que um valor incomum não deve ser classificado automaticamente como erro.

**Resposta:** uma pergunta respondível é: “Como varia a massa corporal entre os pinguins registrados no arquivo?”. Há 342 valores válidos, entre 2.700 g e 6.300 g. Um valor incomum pode refletir variação biológica legítima; antes de classificá-lo como erro, é necessário verificar a unidade, o procedimento de medição, o registro original e sua coerência com as demais variáveis.

## Ciclo didático 2 — População, amostra, representatividade e vieses

### Sortear uma amostra aleatória reproduzível

In [7]:
# Sortear 10 observações sem reposição.
amostra_a = populacao_referencia.sample(
    n=10,
    random_state=42,
)

amostra_a


,studyName,Individual ID,Species,Island,Body Mass (g)
194,PAL0809,N7A1,Gentoo penguin (Pygoscelis papua),Biscoe,4300.0
157,PAL0708,N33A2,Gentoo penguin (Pygoscelis papua),Biscoe,4550.0
225,PAL0809,N55A2,Gentoo penguin (Pygoscelis papua),Biscoe,5200.0
208,PAL0809,N16A1,Gentoo penguin (Pygoscelis papua),Biscoe,4300.0
318,PAL0809,N75A1,Chinstrap penguin (Pygoscelis antarctica),Dream,3550.0
329,PAL0910,N92A2,Chinstrap penguin (Pygoscelis antarctica),Dream,4050.0
319,PAL0809,N75A2,Chinstrap penguin (Pygoscelis antarctica),Dream,3500.0
260,PAL0910,N29A1,Gentoo penguin (Pygoscelis papua),Biscoe,4575.0
114,PAL0910,N61A1,Adelie Penguin (Pygoscelis adeliae),Biscoe,3900.0
220,PAL0809,N53A1,Gentoo penguin (Pygoscelis papua),Biscoe,4700.0


**Tabela 5 - Amostra aleatória A.** Foram sorteadas 10 linhas da população finita de referência. O valor de `random_state` permite reproduzir o mesmo resultado (PANDAS DEVELOPMENT TEAM, 2026). Fonte dos dados: Horst, Hill e Gorman (2020).

In [8]:
# Contar as espécies presentes na amostra A.
composicao_amostra_a = amostra_a["Species"].value_counts()
composicao_amostra_a


Species
Gentoo penguin (Pygoscelis papua)            6
Chinstrap penguin (Pygoscelis antarctica)    3
Adelie Penguin (Pygoscelis adeliae)          1
Name: count, dtype: int64

A seleção é aleatória, mas uma amostra pequena não precisa reproduzir exatamente a composição da população de referência.

### Repetir o procedimento e comparar resultados

In [9]:
# Repetir o sorteio com outra semente.
amostra_b = populacao_referencia.sample(
    n=10,
    random_state=7,
)

amostra_b


,studyName,Individual ID,Species,Island,Body Mass (g)
173,PAL0708,N41A2,Gentoo penguin (Pygoscelis papua),Biscoe,5000.0
30,PAL0708,N21A1,Adelie Penguin (Pygoscelis adeliae),Dream,3250.0
115,PAL0910,N61A2,Adelie Penguin (Pygoscelis adeliae),Biscoe,4075.0
341,PAL0910,N99A2,Chinstrap penguin (Pygoscelis antarctica),Dream,3775.0
141,PAL0910,N80A2,Adelie Penguin (Pygoscelis adeliae),Dream,3475.0
73,PAL0809,N35A2,Adelie Penguin (Pygoscelis adeliae),Torgersen,4150.0
336,PAL0910,N96A1,Chinstrap penguin (Pygoscelis antarctica),Dream,3950.0
328,PAL0910,N92A1,Chinstrap penguin (Pygoscelis antarctica),Dream,3600.0
208,PAL0809,N16A1,Gentoo penguin (Pygoscelis papua),Biscoe,4300.0
28,PAL0708,N18A1,Adelie Penguin (Pygoscelis adeliae),Biscoe,3150.0


**Tabela 6 - Amostra aleatória B.** A mudança de `random_state` produz outra seleção reproduzível de 10 linhas. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

In [10]:
# Contar as espécies presentes na amostra B.
composicao_amostra_b = amostra_b["Species"].value_counts()
composicao_amostra_b


Species
Adelie Penguin (Pygoscelis adeliae)          5
Chinstrap penguin (Pygoscelis antarctica)    3
Gentoo penguin (Pygoscelis papua)            2
Name: count, dtype: int64

In [11]:
# Reunir as contagens para facilitar a comparação.
comparacao_composicoes = pd.DataFrame(
    {
        "População de referência": composicao_referencia,
        "Amostra A": composicao_amostra_a,
        "Amostra B": composicao_amostra_b,
    }
)

# Substituir ausências por zero e usar contagens inteiras.
comparacao_composicoes = comparacao_composicoes.fillna(0).astype(int)

comparacao_composicoes


,População de referência,Amostra A,Amostra B
Species,,,
Adelie Penguin (Pygoscelis adeliae),152,1,5
Chinstrap penguin (Pygoscelis antarctica),68,3,3
Gentoo penguin (Pygoscelis papua),124,6,2


**Tabela 7 - Contagens por espécie na população e nas amostras.** As duas amostras foram obtidas pelo mesmo procedimento, mas apresentam contagens diferentes. Como a população possui 344 observações e cada amostra possui 10, a comparação proporcional será feita na próxima tabela. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).


In [12]:
# Calcular as proporções de cada espécie, em porcentagem.
proporcoes_referencia = composicao_referencia / len(populacao_referencia) * 100
proporcoes_amostra_a = composicao_amostra_a / len(amostra_a) * 100
proporcoes_amostra_b = composicao_amostra_b / len(amostra_b) * 100

# Reunir e arredondar as porcentagens.
comparacao_proporcoes = pd.DataFrame(
    {
        "População de referência (%)": proporcoes_referencia,
        "Amostra A (%)": proporcoes_amostra_a,
        "Amostra B (%)": proporcoes_amostra_b,
    }
).fillna(0).round(1)

comparacao_proporcoes


,População de referência (%),Amostra A (%),Amostra B (%)
Species,,,
Adelie Penguin (Pygoscelis adeliae),44.2,10.0,50.0
Chinstrap penguin (Pygoscelis antarctica),19.8,30.0,30.0
Gentoo penguin (Pygoscelis papua),36.0,60.0,20.0


**Tabela 8 - Proporções das espécies na população e nas amostras.** Adelie corresponde a 44,2% da população de referência, mas representa 10,0% da amostra A e 50,0% da amostra B. Gentoo passa de 36,0% na população para 60,0% na amostra A e 20,0% na amostra B. Essas diferenças ilustram a variabilidade da composição entre amostras aleatórias pequenas. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

**Registro do estudante:** a seleção aleatória produziu uma cópia exata da composição da população? Essas diferenças, isoladamente, comprovam viés?

**Resposta:** não. Nenhuma das amostras reproduziu exatamente a composição da população de referência. As diferenças observadas entre sorteios de dez registros são compatíveis com variabilidade amostral e, isoladamente, não comprovam viés. Para caracterizar viés, seria necessário identificar um mecanismo de seleção que favoreça ou exclua sistematicamente grupos relevantes ao objetivo.


In [13]:
# Calcular a média da massa corporal em cada conjunto.
media_massa_referencia = populacao_referencia["Body Mass (g)"].mean()
media_massa_amostra_a = amostra_a["Body Mass (g)"].mean()
media_massa_amostra_b = amostra_b["Body Mass (g)"].mean()

# Reunir e arredondar as médias.
comparacao_medias = pd.Series(
    {
        "População de referência": media_massa_referencia,
        "Amostra A": media_massa_amostra_a,
        "Amostra B": media_massa_amostra_b,
    },
    name="Média da massa corporal (g)",
).round(1)

comparacao_medias


População de referência    4201.8
Amostra A                  4262.5
Amostra B                  3872.5
Name: Média da massa corporal (g), dtype: float64

**Tabela 9 - Média da massa corporal na população e nas amostras.** A média é 4.201,8 g na população de referência, 4.262,5 g na amostra A e 3.872,5 g na amostra B. A mesma estatística assume valores diferentes porque foi calculada em seleções diferentes. Isso evidencia variabilidade amostral e não demonstra, por si só, a existência de viés. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).


### Analisar um filtro de acordo com o objetivo

### Objetivo A: descrever as três espécies

Suponha que a equipe queira descrever conjuntamente Adelie, Chinstrap e Gentoo, mas filtre apenas Gentoo antes do sorteio. Primeiro, o filtro será criado e aplicado; depois, o subconjunto será amostrado.


In [14]:
# Criar uma condição para selecionar somente Gentoo.
filtro_gentoo = (
    populacao_referencia["Species"]
    == "Gentoo penguin (Pygoscelis papua)"
)

# Aplicar o filtro.
populacao_gentoo = populacao_referencia[filtro_gentoo]

# Verificar o tamanho do subconjunto.
populacao_gentoo.shape


(124, 5)

In [15]:
# Sortear 10 observações dentro do subconjunto Gentoo.
amostra_gentoo = populacao_gentoo.sample(
    n=10,
    random_state=42,
)

amostra_gentoo


,studyName,Individual ID,Species,Island,Body Mass (g)
170,PAL0708,N40A1,Gentoo penguin (Pygoscelis papua),Biscoe,4800.0
194,PAL0809,N7A1,Gentoo penguin (Pygoscelis papua),Biscoe,4300.0
188,PAL0809,N4A1,Gentoo penguin (Pygoscelis papua),Biscoe,4950.0
228,PAL0809,N58A1,Gentoo penguin (Pygoscelis papua),Biscoe,4600.0
205,PAL0809,N14A2,Gentoo penguin (Pygoscelis papua),Biscoe,5000.0
242,PAL0910,N15A1,Gentoo penguin (Pygoscelis papua),Biscoe,4950.0
208,PAL0809,N16A1,Gentoo penguin (Pygoscelis papua),Biscoe,4300.0
274,PAL0910,N43A1,Gentoo penguin (Pygoscelis papua),Biscoe,5200.0
273,PAL0910,N39A2,Gentoo penguin (Pygoscelis papua),Biscoe,5750.0
156,PAL0708,N33A1,Gentoo penguin (Pygoscelis papua),Biscoe,5400.0


**Tabela 10 - Amostra obtida após o filtro por espécie.** O sorteio é aleatório dentro do subconjunto Gentoo, mas Adelie e Chinstrap não podem ser selecionadas. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

In [16]:
# Verificar a relação observada entre espécie e ilha.
especies_por_ilha = pd.crosstab(
    populacao_referencia["Species"],
    populacao_referencia["Island"],
)

especies_por_ilha


Island,Biscoe,Dream,Torgersen
Species,,,
Adelie Penguin (Pygoscelis adeliae),44,56,52
Chinstrap penguin (Pygoscelis antarctica),0,68,0
Gentoo penguin (Pygoscelis papua),124,0,0


**Tabela 11 - Espécies observadas por ilha.** No dataset, Gentoo aparece somente em Biscoe; Chinstrap, somente em Dream; e Adelie, nas três ilhas. Assim, filtrar uma ilha também pode alterar quais espécies podem integrar a seleção. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

Para o **Objetivo A**, o procedimento que produziu a amostra Gentoo apresenta viés de seleção: o procedimento exclui grupos que pertencem à população definida pelo objetivo. A aleatoriedade aplicada depois do filtro não corrige essa exclusão (BRUCE; BRUCE, 2019; ESCOVEDO; MARQUES; KALINOWSKI, 2025).


### Objetivo B: descrever somente Gentoo

Se a pergunta mudar para “como são os pinguins Gentoo registrados no dataset?”, o filtro deixa de ser um erro. A população de interesse passa a ser o subconjunto Gentoo, e a amostra é sorteada dentro do grupo adequado.

O mesmo código pode ser coerente ou inadequado conforme o objetivo da investigação. Por isso, a população de interesse deve ser definida **antes** da seleção dos dados.


### Aplicação orientada

Considere a seguinte solicitação:

> A equipe deseja comparar as três espécies, mas decidiu coletar uma amostra apenas na ilha Biscoe.

Execute a próxima célula e registre, em uma célula Markdown:

1. Qual é a população de interesse declarada?
2. Quais espécies podem aparecer na seleção feita em Biscoe?
3. Qual espécie é excluída e qual fica restrita?
4. A amostra permite comparar as três espécies? Justifique com a saída produzida.


In [17]:
# Selecionar as observações da ilha Biscoe.
filtro_biscoe = populacao_referencia["Island"] == "Biscoe"
populacao_biscoe = populacao_referencia[filtro_biscoe]

# Contar as espécies disponíveis após o filtro.
especies_em_biscoe = populacao_biscoe["Species"].value_counts()
especies_em_biscoe


Species
Gentoo penguin (Pygoscelis papua)      124
Adelie Penguin (Pygoscelis adeliae)     44
Name: count, dtype: int64

**Tabela 12 - Espécies disponíveis após o filtro por Biscoe.** A saída fornece a evidência necessária para avaliar se o procedimento atende ao objetivo de comparar as três espécies. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

**Resposta:** a população de interesse declarada reúne as três espécies registradas no arquivo. Após o filtro por Biscoe, somente Gentoo e Adelie podem aparecer: Chinstrap é excluída e Adelie fica restrita às 44 observações dessa ilha. Portanto, a seleção não permite comparar adequadamente as três espécies e apresenta viés em relação ao objetivo declarado.


## Ciclo didático 3 — Descrição, inferência e alcance das conclusões

Os próximos cálculos descrevem o arquivo. Eles não transformam as 344 observações em uma amostra probabilística de todos os pinguins das espécies.

In [18]:
# Calcular as contagens e as proporções observadas de cada espécie.
contagens_especies = populacao_referencia["Species"].value_counts()
proporcoes_especies = (contagens_especies / len(populacao_referencia) * 100).round(1)

# Calcular a quantidade válida, o menor e o maior valor de massa corporal.
massa_corporal = populacao_referencia["Body Mass (g)"]
amplitude_massa = pd.Series(
    {
        "Quantidade válida": massa_corporal.count(),
        "Mínimo": massa_corporal.min(),
        "Máximo": massa_corporal.max(),
    }
)

# Reunir as descrições em uma única tabela.
descricao_arquivo = pd.concat(
    [contagens_especies, proporcoes_especies, amplitude_massa],
    keys=["Contagem por espécie", "Proporção por espécie (%)", "Massa corporal (g)"],
    names=["Medida", "Categoria"],
).to_frame(name="Valor")

descricao_arquivo


Valor
Medida                    Categoria                                        
Contagem por espécie      Adelie Penguin (Pygoscelis adeliae)         152.0
                          Gentoo penguin (Pygoscelis papua)           124.0
                          Chinstrap penguin (Pygoscelis antarctica)    68.0
Proporção por espécie (%) Adelie Penguin (Pygoscelis adeliae)          44.2
                          Gentoo penguin (Pygoscelis papua)            36.0
                          Chinstrap penguin (Pygoscelis antarctica)    19.8
Massa corporal (g)        Quantidade válida                           342.0
                          Mínimo                                     2700.0
                          Máximo                                     6300.0

**Tabela 13 - Descrição do arquivo por espécie e massa corporal.** As contagens, proporções e amplitudes pertencem às observações disponíveis. Fonte: elaboração própria com dados de Horst, Hill e Gorman (2020).

**Registro do estudante:** escreva uma conclusão descritiva sustentada pela saída e uma generalização que a seleção disponível não permite sustentar.

**Resposta:** no arquivo, Adelie possui 152 registros (44,2%), Gentoo 124 (36,0%) e Chinstrap 68 (19,8%). Entre os 342 valores válidos de massa corporal, os valores variam de 2.700 g a 6.300 g. Esses resultados não permitem afirmar que as mesmas proporções ou a mesma amplitude caracterizam todos os pinguins dessas espécies, pois o arquivo não é uma amostra probabilística dessa população biológica.

## Síntese da prática

- `sample()` sorteia linhas do conjunto informado.
- `random_state` torna o sorteio reproduzível; não torna a amostra automaticamente representativa.
- A composição e as estatísticas calculadas podem variar entre amostras aleatórias.
- **Variabilidade amostral** é a diferença esperada entre resultados de amostras obtidas pelo mesmo procedimento.
- Um filtro redefine quais observações podem ser selecionadas.
- **Viés de seleção** ocorre quando o procedimento exclui sistematicamente grupos necessários ao objetivo.
- Diferença entre amostras não significa automaticamente viés.
- A conclusão deve permanecer dentro do alcance do procedimento de seleção.


## Referências

BRUCE, P.; BRUCE, A. **Estatística prática para cientistas de dados: 50 conceitos essenciais**. Rio de Janeiro: Alta Books, 2019.

ESCOVEDO, T.; MARQUES, T.; KALINOWSKI, M. **Introdução à Estatística para Ciência de Dados: da exploração dos dados à experimentação contínua com exemplos de código em Python e R**. São Paulo: Casa do Código, 2025. Cap. 6.

HORST, A. M.; HILL, A. P.; GORMAN, K. B. **palmerpenguins: Palmer Archipelago (Antarctica) penguin data**. R package version 0.1.0, 2020. Disponível em: <https://allisonhorst.github.io/palmerpenguins/>.

PANDAS DEVELOPMENT TEAM. **pandas documentation: DataFrame.sample**. 2026. Disponível em: <https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html>.
